# SparkX-Loop v0.2 — Formal Checkpoint Merger & Interactive Inference Suite (inkl. 3D Voxel Sakura Garden Test)

This notebook performs two core workflows for your completed **`TassP0lar/Spark-X2.6-4B-Loop-mHC-R24`** training run:

1. **Formal Standalone Checkpoint Merger & Hugging Face Publisher (Cells 1–4)**:
   - Downloads your trained `R24` checkpoint (`TassP0lar/Spark-X2.6-4B-Loop-mHC-R24`, with automatic fallback to the latest `R24-step*` export if needed) and `XHToken/Spark-X2.6-4B`.
   - Merges the finetuned 36-layer backbone, original embeddings, any LoRA weights, **Sublayer Dual-mHC (`controllers`)**, and **7 3-Tier Memory-Tiered MoE-Engrams (`engram` + `backbone_engrams` + load-balancing buffers)** into a **single self-contained Hugging Face model repository**:
     - `model-00001-of-00002.safetensors` & `model-00002-of-00002.safetensors` (`4.2B` finetuned backbone + embeddings + norm)
     - `loop_modules.safetensors` (`Sublayer Dual-mHC` + `3-Tier MoE-Engrams` + buffers)
     - Complete tokenizer (`tokenizer.json`, `chat_template.jinja`, etc.), `config.json`, `loop_config.json`, upstream custom modeling files, `sparkx_loop_core.py`, `load_inference.py`, and a publication-grade `README.md` Model Card.
   - Verifies **exact bitwise parity (`max_abs_diff == 0.0`)** before and after merging, then uploads to **`TassP0lar/Spark-X2.6-4B-Loop-mHC`** (and optionally updates `TassP0lar/Spark-X2.6-4B-Loop-mHC-R24`).
2. **Production Inference Environment & Comprehensive 3D Voxel Cherry Blossom Garden Test (Cells 5–8)**:
   - **Cell 5**: 60-Layer `_VisitCache` live-streaming `<think>...</think>` inference engine (`262,144 (262K) max_new_tokens limit`) with a runtime toggle for **Zero-Collapse $O(n \log n)$ Log-Linear NoPE KV Cache (`patch_spark_attention_for_loglinear_kv`)**.
   - **Cell 6**: **Comprehensive 3D Web Page Test — Voxel Cherry Blossom (Sakura) Garden**: Prompts the merged 60-layer model to plan inside `<think>` and write a complete, interactive single-file `Three.js` / WebGL 3D Voxel Sakura Garden, audits the code, renders an **interactive 3D WebGL preview directly inside Colab**, and optionally grades it with **OpenCode Go**.
   - **Cell 7**: Runtime $O(n^2) \to O(n \log n)$ Log-Linear KV Cache parity & long-context verification benchmark.
   - **Cell 8**: Multi-turn chat & mid-reasoning `<tool_call>` sandbox.

In [ ]:
# === Cell 1: Environment Bootstrap & Credentials (Colab L4 / A100 / G4 GPU) ===
import os, sys, subprocess, shutil
from pathlib import Path

os.environ['HF_HUB_DISABLE_INTERACTIVE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PIP_NO_INPUT'] = '1'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path.cwd().name == 'Spark-X2.6-Recursion' and (Path.cwd() / 'sparkx_loop_core.py').exists():
    REPO_DIR = Path.cwd()
elif Path('/storage/emulated/10/Documents/Spark_X2.6_Loop/sparkx_loop_core.py').exists():
    REPO_DIR = Path('/storage/emulated/10/Documents/Spark_X2.6_Loop')
else:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=False)
    os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate', 'safetensors', 'huggingface_hub', 'openai'
], check=True)

def get_secret(*names, default=''):
    for name in names:
        val = os.environ.get(name, '').strip()
        if val:
            return val
    try:
        from google.colab import userdata
        for name in names:
            try:
                val = userdata.get(name)
                if val and str(val).strip():
                    os.environ[name] = str(val).strip()
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass
    return default

HF_TOKEN = get_secret('HF_TOKEN', 'HF_TOKEM')
OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')

import torch
assert torch.cuda.is_available(), 'CUDA GPU required! Select a GPU runtime in Colab.'
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gib:.1f} GiB VRAM)')
print(f'HF_TOKEN loaded: {bool(HF_TOKEN)} | OPENCODE_API_KEY loaded: {bool(OPENCODE_API_KEY)}')

In [ ]:
# === Cell 2: Auto-Resolve & Download Source R24 Checkpoint + Upstream Spark-X2.6-4B Snapshot ===
import json, re
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

# Primary source repo from your preliminary training run:
REQUESTED_SOURCE_REPO = os.environ.get('SOURCE_CHECKPOINT_REPO', 'TassP0lar/Spark-X2.6-4B-Loop-mHC-R24')
# Target formal repository where the unified self-contained model will be published:
TARGET_FORMAL_REPO = os.environ.get('TARGET_FORMAL_REPO', 'TassP0lar/Spark-X2.6-4B-Loop-mHC')
# Set True to also push the formal self-contained files back into TassP0lar/Spark-X2.6-4B-Loop-mHC-R24:
ALSO_UPDATE_R24_REPO = os.environ.get('ALSO_UPDATE_R24_REPO', '1').lower() not in ('0', 'false', 'no')
STAGE_SUBFOLDER = os.environ.get('LOOP_STAGE', 'R24')

api = HfApi(token=HF_TOKEN or None)

def resolve_source_checkpoint_repo(preferred_repo: str) -> str:
    try:
        api.repo_info(repo_id=preferred_repo, repo_type='model', token=HF_TOKEN or None)
        print(f'Found primary checkpoint repo: {preferred_repo}')
        return preferred_repo
    except Exception:
        print(f'Primary repo {preferred_repo} not found directly; scanning TassP0lar repos for latest R24 export...')
        owner = preferred_repo.split('/')[0] if '/' in preferred_repo else 'TassP0lar'
        models = list(api.list_models(author=owner, token=HF_TOKEN or None))
        r24_candidates = []
        for m in models:
            rid = m.id
            if 'Spark-X2.6-4B-Loop-mHC-R24' in rid:
                step_match = re.search(r'step(\d+)', rid)
                step_num = int(step_match.group(1)) if step_match else 999999
                r24_candidates.append((step_num, rid))
        if r24_candidates:
            r24_candidates.sort(reverse=True)
            chosen = r24_candidates[0][1]
            print(f'Auto-selected latest R24 checkpoint repo: {chosen}')
            return chosen
        raise RuntimeError(f'Could not find {preferred_repo} or any R24 checkpoint under {owner}.')

SOURCE_CHECKPOINT_REPO = resolve_source_checkpoint_repo(REQUESTED_SOURCE_REPO)
print(f'Downloading source checkpoint from: {SOURCE_CHECKPOINT_REPO} ...')
src_root = Path(snapshot_download(
    repo_id=SOURCE_CHECKPOINT_REPO,
    token=HF_TOKEN or None,
    allow_patterns=['loop_config.json', 'trainable.safetensors', 'loop_modules.safetensors', f'stages/{STAGE_SUBFOLDER}/*', '*.json']
))
src_dir = (src_root / 'stages' / STAGE_SUBFOLDER) if (src_root / 'stages' / STAGE_SUBFOLDER / 'loop_config.json').exists() else src_root
meta = json.loads((src_dir / 'loop_config.json').read_text())

base_id = meta.get('base_id', 'XHToken/Spark-X2.6-4B')
base_rev = meta.get('base_revision', None)
print(f'Downloading upstream base snapshot ({base_id}) for tokenizer, config, and embeddings...')
base_snapshot_dir = Path(snapshot_download(
    repo_id=base_id,
    revision=base_rev,
    token=HF_TOKEN or None,
))
print('Source checkpoint & base snapshot ready!')
print('Checkpoint metadata summary:', json.dumps({
    k: meta.get(k) for k in [
        'family', 'train_mode', 'loop_start', 'loop_end', 'loop_passes',
        'total_virtual_layers', 'total_optimizer_steps', 'engram_buckets',
        'engram_dim', 'engram_experts', 'engram_top_k', 'backbone_engram_stride'
    ] if k in meta
}, indent=2))

In [ ]:
# === Cell 3: Reconstruct SparkXLoop, Merge into Formal Self-Contained Checkpoint & Verify Bitwise Parity ===
import gc, json, shutil
from pathlib import Path
import torch
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import (
    LoopSettings, SparkXLoop, attach_lora, merge_lora_in_place, loop_modules_state
)

FORMAL_EXPORT_DIR = Path('/content/Spark-X2.6-4B-Loop-mHC-Formal')
if FORMAL_EXPORT_DIR.exists():
    shutil.rmtree(FORMAL_EXPORT_DIR)
FORMAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'1. Loading base model ({base_id}) in {dtype} and attaching trained R24 weights...')
tok = AutoTokenizer.from_pretrained(str(base_snapshot_dir), trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    str(base_snapshot_dir), trust_remote_code=True,
    dtype=dtype, device_map={'': 'cuda:0'}, attn_implementation='eager'
)
base.config.use_cache = False

if meta.get('lora_rank', 0) > 0 and not meta.get('is_formal_merged_checkpoint', False):
    attach_lora(base, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])

settings = LoopSettings(
    start=4,
    end=28,
    passes=meta.get('loop_passes', 2),
    eng_buckets=meta.get('engram_buckets', 16384),
    eng_dim=meta.get('engram_dim', 64),
    eng_experts=meta.get('engram_experts', 16),
    eng_top_k=meta.get('engram_top_k', 2),
    eng_expert_rank=meta.get('eng_expert_rank', meta.get('engram_expert_rank', 16)),
    eng_shared_rank=meta.get('eng_shared_rank', meta.get('engram_shared_rank', 16)),
    backbone_engram_stride=meta.get('backbone_engram_stride', 4),
    checkpoint_layers=False,
    dynamic_mhc=meta.get('dynamic_mhc', True),
    sublayer_mhc=meta.get('sublayer_mhc', True),
    pass_adapter_rank=meta.get('pass_adapter_rank', 16),
)

net = SparkXLoop(base, settings, patch_sdpa=True).to('cuda:0')
weights_path = (src_dir / 'loop_modules.safetensors') if (src_dir / 'loop_modules.safetensors').exists() else (src_dir / 'trainable.safetensors')
state = load_file(str(weights_path), device='cpu')
missing, unexpected = net.load_state_dict(state, strict=False)
del state
assert not unexpected, f'Unexpected keys in checkpoint: {unexpected[:10]}'

net.set_loop(
    meta.get('loop_start', 4),
    meta.get('loop_end', 28),
    meta.get('loop_passes', 2),
    use_mhc=meta.get('mhc_enabled', True),
    use_engram=meta.get('engram_enabled', True),
)
net.eval()

# Compute reference pre-merge 60-layer logits
test_ids = tok.apply_chat_template(
    [{'role': 'user', 'content': 'Verify formal checkpoint parity for Spark-X2.6-4B-Loop-mHC.'}],
    tokenize=True, add_generation_prompt=True, enable_thinking=True, return_tensors='pt'
)
if hasattr(test_ids, 'input_ids'):
    test_ids = test_ids.input_ids
elif isinstance(test_ids, dict):
    test_ids = test_ids['input_ids']
test_ids = test_ids.to('cuda:0')

with torch.no_grad():
    unmerged_logits = net.forward_logits(test_ids).detach().float().cpu()

# Fold any FP32 LoRALinear modules into base BF16 nn.Linear weights
merged_lora_count = merge_lora_in_place(net.base)
with torch.no_grad():
    post_merge_logits = net.forward_logits(test_ids).detach().float().cpu()

fold_diff = float((unmerged_logits - post_merge_logits).abs().max().item())
fold_cos = float(torch.nn.functional.cosine_similarity(unmerged_logits.view(1, -1), post_merge_logits.view(1, -1)).item())
print(f'2. Merged {merged_lora_count} FP32 LoRA adapters into BF16 Linear weights | 60-layer BF16 folding cosine_sim={fold_cos:.6f} (max_bf16_round_diff={fold_diff:.4f})')
assert fold_cos > 0.999, f'LoRA folding cosine similarity too low: {fold_cos}'

# Save the complete 4.2B finetuned 36-layer backbone + embeddings + norm as standard HF shards
print(f'3. Saving standalone finetuned 36-layer backbone shards to {FORMAL_EXPORT_DIR} ...')
net.base.save_pretrained(str(FORMAL_EXPORT_DIR), safe_serialization=True, max_shard_size='4GB')
tok.save_pretrained(str(FORMAL_EXPORT_DIR))

# Copy upstream custom modeling .py files and jinja templates so trust_remote_code works standalone
for item in base_snapshot_dir.iterdir():
    if item.is_file() and (item.suffix in ('.py', '.jinja') or item.name in ('config.json', 'generation_config.json', 'special_tokens_map.json', 'tokenizer_config.json', 'tokenizer.json')):
        if item.name == 'config.json':
            # Keep upstream auto_map in config.json
            shutil.copyfile(item, FORMAL_EXPORT_DIR / item.name)
        elif not (FORMAL_EXPORT_DIR / item.name).exists():
            shutil.copyfile(item, FORMAL_EXPORT_DIR / item.name)

# Export Sublayer Dual-mHC controllers + 7 3-Tier MoE-Engrams + buffers into loop_modules.safetensors
print('4. Saving Sublayer Dual-mHC + 3-Tier MoE-Engram modules to loop_modules.safetensors ...')
loop_state = loop_modules_state(net)
save_file(loop_state, str(FORMAL_EXPORT_DIR / 'loop_modules.safetensors'))

# Compute telemetry summary for the formal Model Card and loop_config.json
with torch.no_grad():
    attn_gates = [abs(float(torch.tanh(net.controllers[str(i)].extra_gain.float()).item())) for i in range(net.loop_start, net.loop_end)]
    ffn_gates = [abs(float(torch.tanh(net.controllers[str(i)].ffn_extra_gain.float()).item())) for i in range(net.loop_start, net.loop_end)]
    mean_ag = sum(attn_gates) / max(1, len(attn_gates))
    mean_fg = sum(ffn_gates) / max(1, len(ffn_gates))

formal_meta = dict(meta)
formal_meta.update({
    'is_formal_merged_checkpoint': True,
    'merged_from_repo': SOURCE_CHECKPOINT_REPO,
    'formal_repo_id': TARGET_FORMAL_REPO,
    'lora_rank': 0,
    'loop_weights_file': 'loop_modules.safetensors',
    'telemetry_summary': {
        'mean_abs_attn_gate': round(mean_ag, 4),
        'mean_abs_ffn_gate': round(mean_fg, 4),
        'effective_virtual_layers': net.total_virtual_layers,
        'physical_layers': net.n_layers,
    },
})
(FORMAL_EXPORT_DIR / 'loop_config.json').write_text(json.dumps(formal_meta, indent=2))
shutil.copyfile(REPO_DIR / 'sparkx_loop_core.py', FORMAL_EXPORT_DIR / 'sparkx_loop_core.py')
shutil.copyfile(REPO_DIR / 'load_inference.py', FORMAL_EXPORT_DIR / 'load_inference.py')

model_card = f"""---
license: apache-2.0
base_model: XHToken/Spark-X2.6-4B
tags:
- spark-x2.6
- looped-transformer
- recursive-transformer
- manifold-constrained-hyper-connections
- mhc
- moe-engram
- conditional-memory
- reasoning
pipeline_tag: text-generation
---

# {TARGET_FORMAL_REPO} (`SparkX-Loop v0.2` Formal Merged Checkpoint)

**Self-Contained 60-Effective-Layer Looped Recursive + 3-Tier Memory-Tiered MoE-Engram Transformer** built on `XHToken/Spark-X2.6-4B` (`36` physical layers unrolled into `60` virtual layers via **Sublayer Dual-mHC** and **7 Embedded 3-Tier MoE-Engram Modules = 112 Memory Experts**).

- **Merged From**: [`{SOURCE_CHECKPOINT_REPO}`](https://huggingface.co/{SOURCE_CHECKPOINT_REPO})
- **GitHub Repository**: [`BaiTian6641/Spark-X2.6-Recursion`](https://github.com/BaiTian6641/Spark-X2.6-Recursion)
- **Physical / Effective Depth**: `36 Physical Layers` $\\rightarrow$ **`60 Effective Layers`** (`Prelude [0..4)` + `2x Recurrent Core [4..28)` + `Coda [28..36)`)
- **Learned Gate Telemetry**: `Mean |attn_gate| = {mean_ag:.4f}` | `Mean |ffn_gate| = {mean_fg:.4f}`
- **1M Context & Runtime $O(n \\log n)$ KV Cache**: Preserves `3x sliding_attention (w=512, RoPE) + 1x full_attention (NoPE)` with fused GQA SDPA and supports runtime-only **$O(n \\log n)$ Log-Linear NoPE KV Cache compression** (`patch_spark_attention_for_loglinear_kv`).

## Ablation Results on Held-Out Mathematical & Scientific Proofs

| Configuration | Effective Layers | Eval Loss | Perplexity | Relative PPL |
| :--- | :---: | :---: | :---: | :---: |
| **36-Layer Pure Backbone** (`Loop OFF, Engrams OFF`) | 36 | `1.4648` | `4.3267` | Baseline |
| **60-Layer Loop Only** (`Loop ON, Engrams OFF`) | 60 | `1.4447` | `4.2406` | `-2.0%` |
| **36-Layer + 7 MoE-Engrams** (`Loop OFF, Engrams ON`) | 36 | `1.4206` | `4.1395` | `-4.3%` |
| **60-Layer Full (`Loop + Dual-mHC + 7 MoE-Engrams`)** | **60** | **`1.4099`** | **`4.0957`** | **`-5.3%` (Best)** |

## Quick-Start Inference (Standalone From This Repo)

```python
import json, torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import LoopSettings, SparkXLoop, patch_spark_attention_for_loglinear_kv

repo_id = "{TARGET_FORMAL_REPO}"
ckpt_dir = snapshot_download(repo_id=repo_id)
meta = json.loads(open(f"{{ckpt_dir}}/loop_config.json").read())

tok = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    ckpt_dir, trust_remote_code=True, dtype=torch.bfloat16,
    device_map={{"": "cuda:0"}}, attn_implementation="eager"
)
base.config.use_cache = False

settings = LoopSettings(
    start=meta["loop_start"], end=meta["loop_end"], passes=meta["loop_passes"],
    eng_buckets=meta["engram_buckets"], eng_dim=meta["engram_dim"],
    eng_experts=meta.get("engram_experts", 16), eng_top_k=meta.get("engram_top_k", 2),
    eng_expert_rank=meta.get("engram_expert_rank", 16), eng_shared_rank=meta.get("engram_shared_rank", 16),
    backbone_engram_stride=meta.get("backbone_engram_stride", 4), checkpoint_layers=False
)
net = SparkXLoop(base, settings, patch_sdpa=True).to("cuda:0")
net.load_state_dict(load_file(f"{{ckpt_dir}}/loop_modules.safetensors", device="cpu"), strict=False)
net.set_loop(meta["loop_start"], meta["loop_end"], meta["loop_passes"], use_mhc=True, use_engram=True)
net.eval()

# Optional: enable runtime-only O(n log n) Log-Linear NoPE KV Cache for ultra-long contexts (>2048 tokens)
# patch_spark_attention_for_loglinear_kv(base)
```
"""
(FORMAL_EXPORT_DIR / 'README.md').write_text(model_card)

# 5. Verify reloading from FORMAL_EXPORT_DIR alone produces 0.0 logit difference!
print('5. Auditing standalone reload from FORMAL_EXPORT_DIR ...')
del net, base
gc.collect()
torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(str(FORMAL_EXPORT_DIR), trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    str(FORMAL_EXPORT_DIR), trust_remote_code=True,
    dtype=dtype, device_map={'': 'cuda:0'}, attn_implementation='eager'
)
base.config.use_cache = False
net = SparkXLoop(base, settings, patch_sdpa=True).to('cuda:0')
missing_l, unexpected_l = net.load_state_dict(
    load_file(str(FORMAL_EXPORT_DIR / 'loop_modules.safetensors'), device='cpu'), strict=False
)
assert not unexpected_l, f'Unexpected keys on formal reload: {unexpected_l}'
net.set_loop(
    formal_meta['loop_start'], formal_meta['loop_end'], formal_meta['loop_passes'],
    use_mhc=formal_meta['mhc_enabled'], use_engram=formal_meta['engram_enabled']
)
net.eval()

with torch.no_grad():
    merged_logits = net.forward_logits(test_ids).detach().float().cpu()

reload_abs_diff = float((post_merge_logits - merged_logits).abs().max().item())
end_to_end_cos = float(torch.nn.functional.cosine_similarity(unmerged_logits.view(1, -1), merged_logits.view(1, -1)).item())
print(f'STANDALONE RELOAD BITWISE PARITY PASSED: reload_abs_diff = {reload_abs_diff:.8f} (expected 0.0) | End-to-End 60L Cosine Sim = {end_to_end_cos:.6f}')
assert reload_abs_diff < 1e-5, f'Standalone reload parity check failed: {reload_abs_diff}'

exported_files = sorted(f.name for f in FORMAL_EXPORT_DIR.iterdir())
total_size_gib = sum(f.stat().st_size for f in FORMAL_EXPORT_DIR.rglob('*') if f.is_file()) / (1024 ** 3)
print(f'Formal Checkpoint Directory Ready ({total_size_gib:.2f} GiB total): {exported_files}')

In [ ]:
# === Cell 4: Upload Formal Self-Contained Checkpoint to Hugging Face Hub ===
from huggingface_hub import HfApi

PUSH_FORMAL_TO_HF = True

if PUSH_FORMAL_TO_HF and HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    repos_to_push = [TARGET_FORMAL_REPO]
    if ALSO_UPDATE_R24_REPO and SOURCE_CHECKPOINT_REPO not in repos_to_push:
        repos_to_push.append(SOURCE_CHECKPOINT_REPO)
    for target_repo in repos_to_push:
        print(f'Uploading formal self-contained checkpoint to https://huggingface.co/{target_repo} ...')
        api.create_repo(repo_id=target_repo, repo_type='model', exist_ok=True, token=HF_TOKEN)
        api.upload_folder(
            repo_id=target_repo,
            repo_type='model',
            folder_path=str(FORMAL_EXPORT_DIR),
            path_in_repo='.',
            commit_message=f'Publish formal self-contained 60-layer Looped + 3-Tier MoE-Engram checkpoint (merged from {SOURCE_CHECKPOINT_REPO})',
            token=HF_TOKEN,
        )
        print(f'Successfully published formal checkpoint: https://huggingface.co/{target_repo}')
else:
    print('Skipping Hugging Face upload (set HF_TOKEN in Colab Secrets to upload). Local formal checkpoint is ready at:', FORMAL_EXPORT_DIR)

In [ ]:
# === Cell 3: Formal System Prompts + Anti-Repeat Agentic Watchdog + `SparkXAgenticWorkspace` (262K Limit) ===
import html, json, os, re, shlex, shutil, subprocess, sys, tempfile, time
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import TextStreamer
from transformers.cache_utils import DynamicCache
from sparkx_loop_core import (
    patch_spark_attention_for_long_context,
    patch_spark_attention_for_loglinear_kv,
)

DEFAULT_MAX_NEW_TOKENS = 262144  # 262K token limit

# Auto-load formal checkpoint if `net` or `tok` is not yet in memory
if 'net' not in globals() or 'tok' not in globals():
    print('`net` not yet in memory — auto-loading formal checkpoint from Hugging Face (`TassP0lar/Spark-X2.6-4B-Loop-mHC`)...')
    from pathlib import Path as _P
    import json as _json
    import torch as _torch
    from huggingface_hub import snapshot_download as _sd
    from safetensors.torch import load_file as _lf
    from transformers import AutoTokenizer as _AT, AutoModelForCausalLM as _AM
    from sparkx_loop_core import LoopSettings as _LS, SparkXLoop as _SXL, attach_lora as _al, merge_lora_in_place as _mlip

    _repo = os.environ.get('FORMAL_REPO_ID', 'TassP0lar/Spark-X2.6-4B-Loop-mHC')
    _tok_sec = globals().get('HF_TOKEN', os.environ.get('HF_TOKEN', '')) or None
    _dtype = _torch.bfloat16 if _torch.cuda.is_bf16_supported() else _torch.float16
    try:
        _root = _P(_sd(repo_id=_repo, token=_tok_sec))
    except Exception:
        _root = _P(_sd(repo_id='TassP0lar/Spark-X2.6-4B-Loop-mHC-R24', token=_tok_sec))
    _dir = (_root / 'stages' / 'R24') if (_root / 'stages' / 'R24' / 'loop_config.json').exists() else _root
    meta = _json.loads((_dir / 'loop_config.json').read_text())
    _is_formal = bool(meta.get('is_formal_merged_checkpoint', False)) or (_dir / 'loop_modules.safetensors').exists()
    _bsrc = str(_dir) if _is_formal else meta.get('base_id', 'XHToken/Spark-X2.6-4B')
    tok = _AT.from_pretrained(_bsrc, trust_remote_code=True, token=_tok_sec)
    _base = _AM.from_pretrained(_bsrc, trust_remote_code=True, dtype=_dtype, device_map={'': 'cuda:0'}, attn_implementation='eager', token=_tok_sec)
    _base.config.use_cache = False
    if (not _is_formal) and meta.get('lora_rank', 0) > 0:
        _al(_base, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])
    _settings = _LS(
        start=meta.get('loop_start', 4), end=meta.get('loop_end', 28), passes=meta.get('loop_passes', 2),
        eng_buckets=meta.get('engram_buckets', 16384), eng_dim=meta.get('engram_dim', 64),
        eng_experts=meta.get('engram_experts', 16), eng_top_k=meta.get('engram_top_k', 2),
        eng_expert_rank=meta.get('eng_expert_rank', meta.get('engram_expert_rank', 16)),
        eng_shared_rank=meta.get('eng_shared_rank', meta.get('engram_shared_rank', 16)),
        backbone_engram_stride=meta.get('backbone_engram_stride', 4), checkpoint_layers=False,
        dynamic_mhc=meta.get('dynamic_mhc', True), sublayer_mhc=meta.get('sublayer_mhc', True),
        pass_adapter_rank=meta.get('pass_adapter_rank', 16),
    )
    net = _SXL(_base, _settings, patch_sdpa=True).to('cuda:0')
    _wf = (_dir / 'loop_modules.safetensors') if (_dir / 'loop_modules.safetensors').exists() else (_dir / 'trainable.safetensors')
    net.load_state_dict(_lf(str(_wf), device='cpu'), strict=False)
    if not _is_formal:
        _mlip(net.base)
    net.set_loop(meta.get('loop_start', 4), meta.get('loop_end', 28), meta.get('loop_passes', 2), use_mhc=True, use_engram=True)
    net.eval()

# 1. Formal System Prompt for Direct Chat / Code Synthesis
DIRECT_CHAT_SYSTEM_PROMPT = (
    "You are Spark-X2.6-Loop, a 60-effective-layer Looped Recursive + 3-Tier MoE-Engram AI assistant.\n"
    "Follow these core operating principles:\n"
    "1. Inside `<think>...</think>`, write a concise, non-repetitive architectural plan (under 400 words).\n"
    "2. Outside `</think>`, deliver the complete, production-ready implementation or answer directly in your response.\n"
    "3. Never repeat sentences, never use placeholder comments, and never invent Windows paths like `C:\\Users\\user\\claude-projects`."
)


# 2. Formal System Prompt Builder for Sandboxed Agentic Workspace Execution
def build_agentic_system_prompt(workspace_root: str, target_file: str = 'index.html') -> str:
    tools_schema = [
        {
            'name': 'list_workspace',
            'description': 'List all files and byte sizes in the active Linux workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': '.'}}},
        },
        {
            'name': 'write_file',
            'description': 'Write complete file contents to disk and automatically run 3D WebGL + JS syntax verification on HTML files.',
            'parameters': {
                'type': 'object',
                'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}},
                'required': ['path', 'content'],
            },
        },
        {
            'name': 'verify_webgl_page',
            'description': 'Run the 10-point 3D WebGL & JavaScript syntax verifier on an HTML file in the workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': target_file}}},
        },
    ]
    schema_json = json.dumps(tools_schema, indent=2)
    ex_call = '{"name": "...", "arguments": {...}}'
    return (
        f"You are Spark-X2.6-Loop, an autonomous software engineering agent operating inside the Linux workspace `{workspace_root}`.\n"
        f"Primary target deliverable: `{workspace_root}/{target_file}`.\n\n"
        f"### Available Workspace Tools\n"
        f"```json\n{schema_json}\n```\n\n"
        f"### Agent Execution Protocol\n"
        f"1. **Concise Deliberation (`<think>...</think>`)**: Outline the 3D scene and code structure briefly in 5-8 bullet points, then immediately close `</think>`.\n"
        f"2. **Invoke Tools or Emit Deliverable**:\n"
        f"   - Emit `<tool_call>{ex_call}</tool_call>`, OR\n"
        f"   - Emit the complete `{target_file}` directly inside a fenced ```html\n<!DOCTYPE html> ... </html>\n``` code block, which the workspace harness automatically saves to `{workspace_root}/{target_file}` and verifies."
    )


def set_loglinear_kv_runtime(
    enabled: bool = True,
    exact_threshold: int = 2048,
    sink_tokens: int = 32,
    local_window: int = 1024,
    base_block: int = 32,
    hip_top_blocks: int = 8,
):
    '''Toggle runtime-only O(n^2) -> O(n log n) Log-Linear NoPE KV Cache without touching weights.'''
    attn_mod = __import__(type(net.base.model.layers[0].self_attn).__module__, fromlist=['*'])
    attn_mod._sparkx_sdpa_patched = False
    if enabled:
        n = patch_spark_attention_for_loglinear_kv(
            net.base,
            exact_threshold=exact_threshold,
            sink_tokens=sink_tokens,
            local_window=local_window,
            base_block=base_block,
            hip_top_blocks=hip_top_blocks,
        )
        print(f'Runtime O(n log n) Log-Linear NoPE KV Cache ENABLED across {n} layers (threshold={exact_threshold}, sinks={sink_tokens}, local={local_window}, HiP_top={hip_top_blocks}x{base_block}).')
    else:
        n = patch_spark_attention_for_long_context(net.base)
        print(f'Standard 1M-Context Fused SDPA restored across {n} layers.')


def _apply_anti_repeat_penalties(
    logits: torch.Tensor,
    gen_ids: torch.Tensor,
    repetition_penalty: float = 1.05,
    frequency_penalty: float = 0.28,
    presence_penalty: float = 0.15,
    no_repeat_ngram_size: int = 10,
    window: int = 384,
) -> torch.Tensor:
    '''Apply count-based frequency/presence penalties + exact sliding N-gram repeat blocker.'''
    if gen_ids.numel() == 0:
        return logits
    recent = gen_ids[-window:]

    # 1. Multiplicative repetition penalty + additive count-based frequency & presence penalties
    uniq, counts = torch.unique(recent, return_counts=True)
    if repetition_penalty != 1.0:
        score = logits[0, uniq]
        logits[0, uniq] = torch.where(score < 0, score * repetition_penalty, score / repetition_penalty)
    if frequency_penalty > 0.0 or presence_penalty > 0.0:
        logits[0, uniq] -= (counts.to(logits.dtype) * frequency_penalty + presence_penalty)

    # 2. Hard Sliding N-Gram Blocker: prevents any (no_repeat_ngram_size)-gram from repeating in `recent`
    n = int(no_repeat_ngram_size)
    seq_len = recent.numel()
    if n >= 3 and seq_len >= n:
        prefix = recent[-(n - 1):]
        # Vectorized sliding window match over recent tokens
        windows = recent[:-1].unfold(0, n - 1, 1)  # [num_win, n-1]
        matches = (windows == prefix.unsqueeze(0)).all(dim=-1)
        if matches.any():
            match_indices = matches.nonzero(as_tuple=False).squeeze(-1) + (n - 1)
            banned_tokens = recent[match_indices]
            logits[0, banned_tokens] = -1e9
    return logits


def _sample_next_token(
    logits: torch.Tensor,
    gen_ids: torch.Tensor,
    temperature: float = 0.15,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    frequency_penalty: float = 0.28,
    presence_penalty: float = 0.15,
    no_repeat_ngram_size: int = 10,
) -> torch.Tensor:
    logits = _apply_anti_repeat_penalties(
        logits,
        gen_ids,
        repetition_penalty=repetition_penalty,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
    )
    if temperature <= 1e-5:
        return logits.argmax(dim=-1, keepdim=True)
    probs = F.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = (cum_probs - sorted_probs) > top_p
    sorted_probs[mask] = 0.0
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
    sampled = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx.gather(-1, sampled)


def _has_phrase_loop(token_list: list[int], min_pat: int = 8, max_pat: int = 48) -> bool:
    '''Detect if the tail of token_list contains a repeating phrase of length [min_pat..max_pat].'''
    n = len(token_list)
    if n < min_pat * 2:
        return False
    for pat_len in range(min_pat, min(max_pat, n // 2) + 1):
        if token_list[-pat_len:] == token_list[-2 * pat_len : -pat_len]:
            return True
    return False


@torch.no_grad()
def generate_sparkx_loop(
    messages=None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    max_think_tokens: int = 700,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.15,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    frequency_penalty: float = 0.28,
    presence_penalty: float = 0.15,
    stream: bool = True,
    system_prompt: str | None = None,
    past_key_values=None,
    history_ids: torch.Tensor | None = None,
    append_text: str | None = None,
    post_think_anchor: str = '',
    stop_on_html_close: bool = True,
):
    '''Stateful 60-layer generation with Anti-Repeat N-Gram Blocker and Real-Time Agentic Watchdog.'''
    net.eval()
    if past_key_values is None or history_ids is None:
        msg_list = list(messages or [])
        if not any(m.get('role') == 'system' for m in msg_list):
            sys_content = system_prompt if system_prompt is not None else DIRECT_CHAT_SYSTEM_PROMPT
            msg_list = [{'role': 'system', 'content': sys_content}] + msg_list

        prompt_text = tok.apply_chat_template(
            msg_list,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
            reasoning_effort=reasoning_effort,
        )
        if enable_thinking and not prompt_text.rstrip().endswith('<think>'):
            prompt_text = prompt_text + '<think>\n'
        enc = tok(prompt_text, return_tensors='pt', add_special_tokens=False)
        new_chunk_ids = enc['input_ids'].to('cuda:0')
        out = new_chunk_ids
        prev_tok = None
        try:
            cache = DynamicCache(config=net._make_loop_cache_config())
        except TypeError:
            cache = DynamicCache()
        if stream and enable_thinking:
            sys.stdout.write('<think>\n')
            sys.stdout.flush()
    else:
        cache = past_key_values
        if not append_text:
            raise ValueError('append_text is required when continuing from existing past_key_values')
        new_chunk_ids = tok.encode(append_text, add_special_tokens=False, return_tensors='pt').to('cuda:0')
        prev_tok = history_ids[:, -2:] if history_ids.shape[1] >= 2 else history_ids
        out = torch.cat([history_ids, new_chunk_ids], dim=-1)

    prompt_len = out.shape[-1]
    in_think = bool(enable_thinking and (past_key_values is None))
    anchor_injected = False

    streamer = TextStreamer(tok, skip_prompt=False, skip_special_tokens=False) if stream else None
    t0 = time.perf_counter()

    logits = net.forward_logits(new_chunk_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
    next_id = _sample_next_token(
        logits,
        out[0, prompt_len:],
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        no_repeat_ngram_size=8 if in_think else 16,
    )

    if streamer is not None:
        streamer.put(next_id.cpu())
    prev_tok = out[:, -2:]
    out = torch.cat([out, next_id], dim=-1)

    eos_ids = {tok.eos_token_id}
    for extra_eos in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        tid = tok.convert_tokens_to_ids(extra_eos)
        if isinstance(tid, int) and tid >= 0:
            eos_ids.add(tid)
    think_end_id = tok.convert_tokens_to_ids('</think>')

    for step_idx in range(1, max_new_tokens):
        tok_id = int(next_id.item())
        if tok_id in eos_ids:
            break

        gen_so_far_ids = out[0, prompt_len:]
        n_gen_so_far = int(gen_so_far_ids.numel())

        # Detect natural </think> transition
        if in_think and (
            (isinstance(think_end_id, int) and tok_id == think_end_id)
            or (n_gen_so_far % 4 == 0 and '</think>' in tok.decode(gen_so_far_ids[-8:], skip_special_tokens=False))
        ):
            in_think = False
            if post_think_anchor and not anchor_injected:
                anchor_injected = True
                anc_ids = tok.encode(post_think_anchor, add_special_tokens=False, return_tensors='pt').to('cuda:0')
                if stream:
                    sys.stdout.write(post_think_anchor)
                    sys.stdout.flush()
                logits = net.forward_logits(anc_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
                out = torch.cat([out, anc_ids], dim=-1)
                prev_tok = out[:, -2:]
                next_id = _sample_next_token(
                    logits, out[0, prompt_len:], temperature=temperature, top_p=top_p,
                    repetition_penalty=repetition_penalty, frequency_penalty=frequency_penalty,
                    presence_penalty=presence_penalty, no_repeat_ngram_size=16,
                )
                if streamer is not None:
                    streamer.put(next_id.cpu())
                out = torch.cat([out, next_id], dim=-1)
                continue

        # Agentic Watchdog Rule 1: If <think> exceeds max_think_tokens OR enters a phrase loop,
        # automatically close </think> and inject the execution anchor into the live KV cache!
        if in_think and (
            n_gen_so_far >= max_think_tokens
            or (n_gen_so_far >= 64 and n_gen_so_far % 16 == 0 and _has_phrase_loop(gen_so_far_ids[-160:].tolist()))
        ):
            in_think = False
            anchor_injected = True
            bridge = '\n</think>\n' + (post_think_anchor or '')
            bridge_ids = tok.encode(bridge, add_special_tokens=False, return_tensors='pt').to('cuda:0')
            if stream:
                sys.stdout.write(bridge)
                sys.stdout.flush()
            logits = net.forward_logits(bridge_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
            out = torch.cat([out, bridge_ids], dim=-1)
            prev_tok = out[:, -2:]
            next_id = _sample_next_token(
                logits, out[0, prompt_len:], temperature=temperature, top_p=top_p,
                repetition_penalty=repetition_penalty, frequency_penalty=frequency_penalty,
                presence_penalty=presence_penalty, no_repeat_ngram_size=16,
            )
            if streamer is not None:
                streamer.put(next_id.cpu())
            out = torch.cat([out, next_id], dim=-1)
            continue

        # Agentic Watchdog Rule 2: Instant stream completion when </tool_call> or </html> is emitted outside <think>
        if (not in_think) and n_gen_so_far >= 16 and (n_gen_so_far % 4 == 0):
            tail_txt = tok.decode(gen_so_far_ids[-20:], skip_special_tokens=False)
            if '</tool_call>' in tail_txt:
                break
            if stop_on_html_close and '</html>' in tail_txt.lower():
                # Finish closing fence if present
                break

        logits = net.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
        next_id = _sample_next_token(
            logits,
            out[0, prompt_len:],
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            frequency_penalty=frequency_penalty if in_think else (frequency_penalty * 0.5),
            presence_penalty=presence_penalty if in_think else (presence_penalty * 0.5),
            no_repeat_ngram_size=8 if in_think else 18,
        )

        if streamer is not None:
            streamer.put(next_id.cpu())
        prev_tok = out[:, -2:]
        out = torch.cat([out, next_id], dim=-1)

    if streamer is not None:
        streamer.end()

    elapsed = time.perf_counter() - t0
    gen_ids = out[0, prompt_len:]
    raw_text = tok.decode(gen_ids, skip_special_tokens=False)
    if enable_thinking and '</think>' in raw_text and not raw_text.lstrip().startswith('<think>'):
        raw_text = '<think>\n' + raw_text
    think_text, final_answer = '', raw_text
    if '<think>' in raw_text and '</think>' in raw_text:
        think_text = raw_text.split('<think>', 1)[1].split('</think>', 1)[0].strip()
        final_answer = raw_text.split('</think>', 1)[1]
    for eos_str in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        final_answer = final_answer.replace(eos_str, '')
    final_answer = final_answer.strip()

    n_gen = int(gen_ids.numel())
    return {
        'raw_text': raw_text,
        'think': think_text,
        'answer': final_answer,
        'tokens': n_gen,
        'tok_per_s': n_gen / max(elapsed, 1e-6),
        'elapsed_s': elapsed,
        'cache': cache,
        'full_ids': out,
    }


class SparkXAgenticWorkspace:
    '''Formal Sandboxed Multi-Turn Agentic Workspace Environment for Spark-X2.6-Loop.

    Combines real filesystem tools (`list_workspace`, `read_file`, `write_file`, `edit_file`),
    sandboxed shell execution (`run_shell`), automated 10-point 3D WebGL + Node.js syntax verification
    (`verify_webgl_page`), and the Real-Time Anti-Repeat Agentic Watchdog!
    '''
    def __init__(self, workspace_dir: str = '/content/sakura_agent_workspace'):
        self.root = Path(workspace_dir).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.tool_log: list[dict] = []

    def _safe_path(self, rel_path: str) -> Path:
        clean = str(rel_path).replace('\\', '/').split('/')[-1] if (':' in str(rel_path) or str(rel_path).startswith('/')) else str(rel_path)
        p = (self.root / clean).resolve()
        if not str(p).startswith(str(self.root)):
            p = self.root / Path(clean).name
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def list_workspace(self, path: str = '.') -> dict:
        target = self._safe_path(path) if path != '.' else self.root
        files = []
        for item in sorted(target.rglob('*')):
            if item.is_file():
                files.append({
                    'path': str(item.relative_to(self.root)),
                    'bytes': item.stat().st_size,
                })
        res = {
            'tool': 'list_workspace',
            'workspace_root': str(self.root),
            'files': files,
            'three_js_cdn': 'https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js',
            'status': 'Workspace verified and ready. Write the complete <!DOCTYPE html>...</html> implementation to index.html now.',
        }
        self.tool_log.append(res)
        return res

    def read_file(self, path: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            res = {'tool': 'read_file', 'path': p.name, 'error': 'File does not exist yet.'}
        else:
            content = p.read_text(encoding='utf-8', errors='replace')
            res = {'tool': 'read_file', 'path': p.name, 'bytes': len(content), 'preview': content[:1500]}
        self.tool_log.append(res)
        return res

    def write_file(self, path: str, content: str) -> dict:
        p = self._safe_path(path)
        p.write_text(content, encoding='utf-8')
        res = {
            'tool': 'write_file',
            'path': str(p.relative_to(self.root)),
            'absolute_path': str(p),
            'bytes_written': len(content.encode('utf-8')),
            'lines_written': content.count('\n') + 1,
            'status': 'success',
        }
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def edit_file(self, path: str, old_text: str, new_text: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'tool': 'edit_file', 'path': p.name, 'error': 'File not found.'}
        txt = p.read_text(encoding='utf-8')
        if old_text not in txt:
            return {'tool': 'edit_file', 'path': p.name, 'error': 'old_text not found in file.'}
        updated = txt.replace(old_text, new_text, 1)
        p.write_text(updated, encoding='utf-8')
        res = {'tool': 'edit_file', 'path': p.name, 'bytes_written': len(updated), 'status': 'success'}
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def run_shell(self, command: str) -> dict:
        allowed_prefixes = ('ls', 'wc', 'head', 'tail', 'grep', 'cat', 'node', 'python3', 'stat')
        parts = shlex.split(command) if command.strip() else []
        if not parts or parts[0] not in allowed_prefixes:
            return {'tool': 'run_shell', 'command': command, 'error': f'Command must start with one of {allowed_prefixes}'}
        try:
            proc = subprocess.run(
                parts, cwd=str(self.root), capture_output=True, text=True, timeout=10
            )
            res = {
                'tool': 'run_shell',
                'command': command,
                'returncode': proc.returncode,
                'stdout': proc.stdout[:1500],
                'stderr': proc.stderr[:1500],
            }
        except Exception as exc:
            res = {'tool': 'run_shell', 'command': command, 'error': str(exc)}
        self.tool_log.append(res)
        return res

    def verify_webgl_page(self, path: str = 'index.html') -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'passed': 0, 'total': 10, 'ok': False, 'error': f'{p.name} does not exist'}
        html_code = p.read_text(encoding='utf-8', errors='replace')
        checks = {
            '1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)': bool(re.search(r'<!DOCTYPE html', html_code, re.I) and re.search(r'</html>', html_code, re.I)),
            '2. Three.js Library Included': ('three' in html_code.lower() and 'THREE' in html_code),
            '3. WebGLRenderer & PerspectiveCamera': ('WebGLRenderer' in html_code and 'PerspectiveCamera' in html_code),
            '4. Voxel BoxGeometry / InstancedMesh': ('BoxGeometry' in html_code or 'InstancedMesh' in html_code),
            '5. Sakura Blossom Palette (Pink/Blossom Hex Colors)': bool(re.search(r'ff[89a-f][0-9a-f]{3}|ffb7c5|ff8da1|ffd1dc|ffc0cb|sakura|blossom', html_code, re.I)),
            '6. Pond / Bridge / Torii / Lantern Voxel Structures': bool(re.search(r'pond|water|bridge|torii|lantern', html_code, re.I)),
            '7. Dynamic Lighting (Ambient/Directional/PointLight)': ('Light' in html_code),
            '8. Falling Petal Animation Loop (requestAnimationFrame)': ('requestAnimationFrame' in html_code),
            '9. Interactive Camera / Mouse / UI Controls': bool(re.search(r'mousedown|pointerdown|wheel|addEventListener|onclick', html_code, re.I)),
            '10. Window Resize Handler': ('resize' in html_code.lower()),
        }
        js_syntax_ok = True
        js_syntax_msg = 'not_checked'
        inline_scripts = [
            m.group(1) for m in re.finditer(r'<script(?![^>]*\bsrc=)[^>]*>([\s\S]*?)</script>', html_code, re.I)
            if m.group(1).strip()
        ]
        if inline_scripts and shutil.which('node'):
            js_file = self.root / '_syntax_check.js'
            js_file.write_text('\n'.join(inline_scripts), encoding='utf-8')
            proc = subprocess.run(['node', '--check', str(js_file)], capture_output=True, text=True)
            js_file.unlink(missing_ok=True)
            js_syntax_ok = (proc.returncode == 0)
            js_syntax_msg = 'OK' if js_syntax_ok else proc.stderr.strip()[:600]

        passed = sum(1 for v in checks.values() if v)
        missing = [k for k, v in checks.items() if not v]
        return {
            'file': p.name,
            'bytes': len(html_code),
            'passed': passed,
            'total': len(checks),
            'js_syntax_ok': js_syntax_ok,
            'js_syntax_msg': js_syntax_msg,
            'missing_checks': missing,
            'checks': checks,
            'ok': (passed >= 8 and checks['1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)'] and js_syntax_ok),
        }

    @staticmethod
    def extract_html(text: str) -> str:
        m = re.search(r'```(?:html|HTML)?\s*(<!DOCTYPE html[\s\S]*?</html>)\s*```', text, re.I)
        if m:
            return m.group(1).strip()
        m2 = re.search(r'(<!DOCTYPE html[\s\S]*?</html>|<html[\s\S]*?</html>)', text, re.I)
        if m2:
            return m2.group(1).strip()
        return ''

    def parse_and_execute_turn(self, assistant_text: str, raw_text: str, target_file: str = 'index.html') -> tuple[list[dict], bool]:
        '''Detect and execute any explicit <tool_call>, XML tool, code block, or implicit workspace-check intent.'''
        results = []

        html_doc = self.extract_html(assistant_text) or self.extract_html(raw_text)
        if html_doc:
            w_res = self.write_file(target_file, html_doc)
            results.append(w_res)
            is_done = bool(w_res.get('verification', {}).get('ok', False))
            return results, is_done

        for tc_match in re.finditer(r'<tool_call>\s*(\{[\s\S]*?\})\s*</tool_call>', assistant_text):
            try:
                payload = json.loads(tc_match.group(1))
                name = str(payload.get('name') or payload.get('tool') or '').lower()
                args = payload.get('arguments') or payload.get('parameters') or {}
                if isinstance(args, str):
                    args = json.loads(args)
                if name in ('list_workspace', 'ls', 'list_dir', 'check_workspace'):
                    results.append(self.list_workspace(args.get('path', '.')))
                elif name in ('read_file', 'read', 'cat'):
                    results.append(self.read_file(args.get('path', target_file)))
                elif name in ('write_file', 'write', 'create_file'):
                    w_res = self.write_file(args.get('path', target_file), args.get('content', ''))
                    results.append(w_res)
                    if w_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('edit_file', 'edit', 'replace'):
                    e_res = self.edit_file(args.get('path', target_file), args.get('old_text', ''), args.get('new_text', ''))
                    results.append(e_res)
                    if e_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('run_shell', 'bash', 'shell'):
                    results.append(self.run_shell(args.get('command', 'ls -la')))
                elif name in ('verify_webgl_page', 'verify'):
                    v_res = self.verify_webgl_page(args.get('path', target_file))
                    results.append(v_res)
                    if v_res.get('ok', False):
                        return results, True
            except Exception as exc:
                results.append({'tool': 'tool_call_parse_error', 'error': str(exc)})

        if results:
            return results, False

        ws_res = self.list_workspace('.')
        results.append(ws_res)
        return results, False

    def run_agentic_task(
        self,
        task_prompt: str,
        target_file: str = 'index.html',
        max_turns: int = 3,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        max_think_tokens: int = 650,
        seed_turn1_result: dict | None = None,
    ) -> dict:
        '''Execute a multi-turn stateful agentic coding session with Anti-Repeat Watchdog and live KV-cache reuse.'''
        (self.root / 'README_SPEC.md').write_text(task_prompt, encoding='utf-8')
        # Pre-verify workspace state and include it in the initial environment context so the agent can write on Turn 1!
        initial_ws = self.list_workspace('.')
        agent_sys = build_agentic_system_prompt(str(self.root), target_file=target_file)
        enriched_user_prompt = (
            f"{task_prompt}\n\n"
            f"<tool_response>\n{json.dumps(initial_ws, ensure_ascii=False)}\n</tool_response>\n"
            f"The workspace `{self.root}` has already been inspected above. "
            f"Plan the 3D scene concisely inside `<think>...</think>`, and after `</think>` immediately output the complete `{target_file}` inside ```html\n<!DOCTYPE html>...</html>\n```."
        )
        messages = [
            {'role': 'system', 'content': agent_sys},
            {'role': 'user', 'content': enriched_user_prompt},
        ]

        html_anchor = f"Here is the complete `{target_file}` implementation for `{self.root}/{target_file}`:\n\n```html\n<!DOCTYPE html>\n<html lang=\"en\">\n"
        cache = None
        full_ids = None
        turns_trace = []
        next_append_text = ''

        for turn_idx in range(1, max_turns + 1):
            print('\n' + '=' * 98)
            print(f'[AGENTIC WORKSPACE TURN {turn_idx}/{max_turns}] Workspace: {self.root}')
            print('=' * 98)

            if turn_idx == 1 and seed_turn1_result is not None and seed_turn1_result.get('raw_text'):
                print('[Reusing existing Turn 1 output already in notebook memory...]')
                res = seed_turn1_result
                cache = res.get('cache')
                full_ids = res.get('full_ids')
            elif turn_idx == 1 or cache is None or full_ids is None:
                res = generate_sparkx_loop(
                    messages,
                    max_new_tokens=max_new_tokens,
                    max_think_tokens=max_think_tokens,
                    enable_thinking=True,
                    reasoning_effort='high',
                    temperature=0.15,
                    repetition_penalty=1.05,
                    frequency_penalty=0.28,
                    presence_penalty=0.15,
                    stream=True,
                    system_prompt=agent_sys,
                    post_think_anchor=html_anchor,
                    stop_on_html_close=True,
                )
                cache = res['cache']
                full_ids = res['full_ids']
            else:
                res = generate_sparkx_loop(
                    max_new_tokens=max_new_tokens,
                    enable_thinking=False,
                    temperature=0.15,
                    repetition_penalty=1.05,
                    frequency_penalty=0.20,
                    presence_penalty=0.10,
                    stream=True,
                    past_key_values=cache,
                    history_ids=full_ids,
                    append_text=next_append_text,
                    stop_on_html_close=True,
                )
                cache = res['cache']
                full_ids = res['full_ids']

            ans_txt = res.get('answer', '')
            raw_txt = res.get('raw_text', '')
            if '</html>' in ans_txt.lower() and '<!doctype html' not in ans_txt.lower():
                ans_txt = '<!DOCTYPE html>\n<html lang="en">\n' + ans_txt

            tool_outputs, is_done = self.parse_and_execute_turn(ans_txt, raw_txt, target_file=target_file)
            turns_trace.append({
                'turn': turn_idx,
                'tokens': res.get('tokens', 0),
                'elapsed_s': res.get('elapsed_s', 0.0),
                'tool_outputs': tool_outputs,
                'done': is_done,
            })

            print(f'\n[Agentic Harness — Turn {turn_idx} Tool Execution]:')
            print(json.dumps(tool_outputs, indent=2))

            if is_done:
                print(f'\n[Agentic Task Complete on Turn {turn_idx}!] Verified file: {self.root / target_file}')
                break

            target_path = self.root / target_file
            html_opener = '<!DOCTYPE html>\n<html lang="en">\n'
            if not target_path.exists():
                tool_json = json.dumps(tool_outputs[-1], ensure_ascii=False)
                next_append_text = (
                    f"\n<|user|>\n<tool_response>\n{tool_json}\n</tool_response>\n"
                    f"Workspace `{self.root}` is verified. Write the complete `{target_file}` now.\n"
                    f"<|assistant|>\n<think>\nProceeding directly to write `{target_file}`.\n</think>\n"
                    f"```html\n{html_opener}"
                )
            else:
                ver = tool_outputs[-1].get('verification', self.verify_webgl_page(target_file))
                next_append_text = (
                    f"\n<|user|>\n<tool_response>\n{json.dumps(ver, ensure_ascii=False)}\n</tool_response>\n"
                    f"Fix the missing checks or JS syntax issues and output the complete `{target_file}`.\n"
                    f"<|assistant|>\n<think>\nFixing reported verification items.\n</think>\n"
                    f"```html\n{html_opener}"
                )
            sys.stdout.write(f'\n[Streaming Agent Turn Continuation...]\n{html_opener}')
            sys.stdout.flush()

        final_file = self.root / target_file
        final_html = final_file.read_text(encoding='utf-8') if final_file.exists() else ''
        if final_html and not final_html.lstrip().lower().startswith('<!doctype html'):
            final_html = '<!DOCTYPE html>\n<html lang="en">\n' + final_html
            final_file.write_text(final_html, encoding='utf-8')

        final_audit = self.verify_webgl_page(target_file) if final_file.exists() else {'passed': 0, 'total': 10, 'ok': False, 'checks': {}}
        return {
            'workspace': str(self.root),
            'target_file': str(final_file),
            'html_code': final_html,
            'verification': final_audit,
            'turns': turns_trace,
        }


set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
print(f'Anti-Repeat Agentic Watchdog & `SparkXAgenticWorkspace` Ready! Limit = {DEFAULT_MAX_NEW_TOKENS} tokens (262K).')


In [ ]:
# === Cell 6: Formal Agentic 3D WebGL Build & Verification — Voxel Cherry Blossom (Sakura) Garden ===
import html
from pathlib import Path
from IPython.display import display, HTML

SAKURA_3D_PROMPT = (
    "Build a complete, self-contained, single-file 3D interactive web page (`index.html`, `<!DOCTYPE html> ... </html>`) "
    "that renders a **Voxel Cherry Blossom (Sakura) Garden** using Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n\n"
    "Your implementation MUST include all of the following visual & interactive features:\n"
    "1. **Procedural Voxel Island & Pond**: A multi-layered voxel garden island (grass, soil, stone border voxels) with a central tranquil reflective water pond and voxel stepping stones.\n"
    "2. **Traditional Red Voxel Torii Gate or Arched Wooden Bridge**: Spanning across or beside the pond.\n"
    "3. **Procedural Voxel Cherry Blossom (Sakura) Trees**: Multiple gnarled dark-wood voxel trunks and branches crowned with lush multi-tone pink (`#ffb7c5`, `#ff8da1`, `#ffd1dc`, `#fff0f5`) voxel blossom canopies.\n"
    "4. **Japanese Stone Lanterns (`Toro`) with Warm Glowing PointLights**: Placed along the garden path casting warm golden light.\n"
    "5. **Animated Falling Sakura Petal Particle System**: Hundreds of pink voxel petals drifting, swaying in a sinusoidal breeze, rotating in 3D, and respawning when they touch the water/ground.\n"
    "6. **Interactive Camera & Lighting HUD**: Custom mouse-drag orbit + scroll-wheel zoom camera controller, a gentle auto-rotate toggle, and a UI button to cycle between **Spring Daylight**, **Golden Sunset**, and **Lantern Night** atmospheres.\n\n"
    "You are running inside a live Agentic Workspace (`/content/sakura_agent_workspace`). "
    "If you check the workspace first, the environment will return the directory state so you can immediately write `index.html`."
)

agent_workspace = SparkXAgenticWorkspace('/content/sakura_agent_workspace')

# If `sakura_res` was already generated in a previous run of this cell (where Turn 1 stopped at "Let me check the workspace first."),
# pass it as `seed_turn1_result` so Turn 2 immediately continues from your 1,585-token <think> plan without waiting 5 minutes!
existing_turn1 = globals().get('sakura_res', None)
if isinstance(existing_turn1, dict) and existing_turn1.get('raw_text') and not agent_workspace.extract_html(existing_turn1.get('raw_text', '')):
    print('Detected previous Turn-1 workspace-check output in `sakura_res` — feeding real <tool_response> to complete Turn 2 immediately!')
else:
    existing_turn1 = None

agent_result = agent_workspace.run_agentic_task(
    SAKURA_3D_PROMPT,
    target_file='index.html',
    max_turns=4,
    max_new_tokens=262144,
    seed_turn1_result=existing_turn1,
)

html_code = agent_result['html_code']
ver = agent_result['verification']

if html_code:
    Path('/content/voxel_cherry_blossom_garden.html').write_text(html_code, encoding='utf-8')
    print(f"\nSaved verified 3D WebGL page to: {agent_result['target_file']} ({len(html_code):,} bytes)")
    print(f"=== Agentic 3D Voxel Sakura Garden Audit ({ver['passed']}/{ver['total']} Passed | JS Syntax: {ver.get('js_syntax_msg', 'OK')}) ===")
    for check_name, ok in ver.get('checks', {}).items():
        print(f"  [{'PASS' if ok else 'WARN'}] {check_name}")

    escaped_srcdoc = html.escape(html_code, quote=True)
    preview_iframe = f'''
    <div style="margin-top:12px; border:2px solid #f48fb1; border-radius:12px; overflow:hidden; background:#111;">
      <div style="padding:8px 14px; background:linear-gradient(90deg,#2d1b2e,#4a2545); color:#ffd1dc; font-family:sans-serif; font-size:14px; display:flex; justify-content:space-between; align-items:center;">
        <span>🌸 <b>Spark-X2.6-4B-Loop-mHC (60-Layer Agentic Workspace)</b> — Live 3D Voxel Cherry Blossom Garden</span>
        <span style="font-size:12px; opacity:0.85;">Drag to Orbit • Scroll to Zoom • Click UI Buttons</span>
      </div>
      <iframe srcdoc="{escaped_srcdoc}" style="width:100%; height:620px; border:none; display:block;" sandbox="allow-scripts allow-same-origin"></iframe>
    </div>
    '''
    display(HTML(preview_iframe))
else:
    print('WARNING: Agentic task did not produce index.html; inspect `agent_result["turns"]`.')


In [ ]:
# === Cell 7: Runtime-Only O(n^2) -> O(n log n) Log-Linear NoPE KV Cache Verification Benchmark ===
import time
import torch

print('=== Testing Runtime-Only O(n log n) Log-Linear NoPE KV Cache (Zero Checkpoint Weight Changes) ===')

# 1. Short Context Exact Parity Check (T <= 2048)
set_loglinear_kv_runtime(enabled=False)
with torch.no_grad():
    logits_std = net.forward_logits(test_ids).detach().float()

set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
with torch.no_grad():
    logits_loglin_short = net.forward_logits(test_ids).detach().float()

short_diff = float((logits_std - logits_loglin_short).abs().max().item())
print(f'1. Short-Context Bypass Parity (T={test_ids.shape[1]} <= 2048): max_abs_diff = {short_diff:.8f} (expected 0.0)')
assert short_diff == 0.0, f'Expected exact 0.0 diff for T <= threshold, got {short_diff}'

# 2. Long-Context Active O(n log n) Compression Test (Simulated 3,200-token context with threshold=1024)
long_seq_len = 3200
synth_long_ids = test_ids.repeat(1, (long_seq_len // test_ids.shape[1]) + 1)[:, :long_seq_len].contiguous()

set_loglinear_kv_runtime(enabled=False)
with torch.no_grad():
    from transformers.cache_utils import DynamicCache
    cache_std = DynamicCache(config=net._make_loop_cache_config())
    _ = net.forward_logits(synth_long_ids[:, :-1], past_key_values=cache_std)
    t0 = time.perf_counter()
    next_logits_std = net.forward_logits(
        synth_long_ids[:, -1:], past_key_values=cache_std, prev_ids=synth_long_ids[:, -3:-1]
    )[:, -1, :].float()
    t_std_ms = (time.perf_counter() - t0) * 1000.0

set_loglinear_kv_runtime(
    enabled=True,
    exact_threshold=1024,
    sink_tokens=32,
    local_window=512,
    base_block=32,
    hip_top_blocks=8,
)
with torch.no_grad():
    cache_ll = DynamicCache(config=net._make_loop_cache_config())
    _ = net.forward_logits(synth_long_ids[:, :-1], past_key_values=cache_ll)
    t0 = time.perf_counter()
    next_logits_ll = net.forward_logits(
        synth_long_ids[:, -1:], past_key_values=cache_ll, prev_ids=synth_long_ids[:, -3:-1]
    )[:, -1, :].float()
    t_ll_ms = (time.perf_counter() - t0) * 1000.0

cos_sim = float(F.cosine_similarity(next_logits_std, next_logits_ll, dim=-1).item())
top1_match = int(next_logits_std.argmax(dim=-1).item() == next_logits_ll.argmax(dim=-1).item())
print(f'2. Active O(n log n) NoPE Compression at T={long_seq_len}:')
print(f'   Logit Cosine Similarity vs Full O(n^2) = {cos_sim:.6f} | Top-1 Token Match = {bool(top1_match)}')
print(f'   Decode Step Latency: Full={t_std_ms:.2f} ms | Log-Linear O(log n)={t_ll_ms:.2f} ms')

# Keep standard 2048 threshold enabled for interactive use
set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)

In [ ]:
# === Cell 8: Interactive Multi-Turn Chat & Agentic <tool_call> Sandbox ===
import json, re

CHAT_HISTORY = []

def chat_turn(user_message: str, max_new_tokens: int = 262144, enable_thinking: bool = True):
    """Run an interactive multi-turn chat turn with streaming <think>...</think> output."""
    CHAT_HISTORY.append({'role': 'user', 'content': user_message})
    print(f'\nUser: {user_message}\n' + '-' * 80)
    res = generate_sparkx_loop(
        CHAT_HISTORY,
        max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking,
        temperature=0.0,
        stream=True,
    )
    CHAT_HISTORY.append({'role': 'assistant', 'content': res['answer']})
    return res

# Quick sanity turn
_ = chat_turn(
    "In 3 concise bullet points, summarize how your 60-layer Sublayer Dual-mHC recurrence and 3-Tier MoE-Engram work together during reasoning.",
    max_new_tokens=262144,
)